# Bird Acoustics Classifier — Data Pipeline

End-to-end pipeline from raw audio to training-ready spectrogram images.

| Step | Description |
|------|-------------|
| 1 | **Setup** — environment detection (local / Colab) |
| 2 | **Configuration** — load parameters from `config/default.yaml` |
| 3 | **Download** — fetch `.mp3` recordings from Xeno-canto API |
| 4 | **Preprocessing** — convert audio to mel-spectrogram PNG tiles |
| 5 | **Summary** — dataset statistics and visual inspection |

> **Run All** (`Runtime → Run all` on Colab, `Kernel → Restart & Run All` locally) executes the full pipeline in one shot.

---
## 1. Setup

In [ ]:
import sys, os

BRANCH = "claude/setup-project-structure-lVRTH"

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    repo = '/content/bird-acoustics-classifier'
    if not os.path.exists(repo):
        !git clone -q -b {BRANCH} https://github.com/danort92/bird-acoustics-classifier.git {repo}
    else:
        !git -C {repo} fetch -q origin {BRANCH} && git -C {repo} checkout -q {BRANCH} && git -C {repo} pull -q
    %cd /content/bird-acoustics-classifier
    !pip install -q -r requirements.txt

    # Symlink data dirs to Drive so files survive session restarts
    for subdir in ['data/raw', 'data/processed']:
        drive_path = f'/content/drive/MyDrive/bird-acoustics-classifier/{subdir}'
        os.makedirs(drive_path, exist_ok=True)
        if os.path.islink(subdir) or os.path.isdir(subdir):
            !rm -rf {subdir}
        !ln -s {drive_path} {subdir}

    sys.path.insert(0, repo)
    print(f'Colab ready (branch: {BRANCH}). Data will be saved to Google Drive.')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    sys.path.insert(0, os.getcwd())
    print('Local environment. Working directory:', os.getcwd())

---
## 2. Configuration

Parameters are loaded from `config/default.yaml`. Override any value in the cell below.

### Available species — Alpine zone (Italy / Austria / Switzerland)

| # | Scientific name | Common name | Habitat |
|---|----------------|-------------|---------|
| 1 | *Turdus torquatus* | Ring ouzel | Rocky slopes, high-altitude forests |
| 2 | *Phoenicurus ochruros* | Black redstart | Rocky terrain, mountain villages |
| 3 | *Prunella collaris* | Alpine accentor | High rocky areas above treeline |
| 4 | *Pyrrhocorax graculus* | Yellow-billed chough | Alpine cliffs and glaciers |
| 5 | *Pyrrhocorax pyrrhocorax* | Red-billed chough | Alpine meadows and cliffs |
| 6 | *Tichodroma muraria* | Wallcreeper | Vertical rock faces |
| 7 | *Anthus spinoletta* | Water pipit | Alpine meadows and streams |
| 8 | *Montifringilla nivalis* | White-winged snowfinch | Above treeline, snowfields |
| 9 | *Lagopus muta* | Rock ptarmigan | High alpine tundra |
| 10 | *Dryocopus martius* | Black woodpecker | Subalpine conifer forests |
| 11 | *Tetrao urogallus* | Western capercaillie | Old-growth conifer forests |
| 12 | *Picoides tridactylus* | Three-toed woodpecker | Spruce forests |
| 13 | *Loxia curvirostra* | Common crossbill | Conifer forests |
| 14 | *Nucifraga caryocatactes* | Spotted nutcracker | Mountain conifer forests |
| 15 | *Regulus ignicapilla* | Firecrest | Mixed mountain forests |
| 16 | *Cinclus cinclus* | White-throated dipper | Alpine streams and torrents |
| 17 | *Ficedula albicollis* | Collared flycatcher | Deciduous mountain forests |
| 18 | *Saxicola rubetra* | Whinchat | Subalpine meadows |
| 19 | *Emberiza cia* | Rock bunting | Rocky slopes with sparse vegetation |
| 20 | *Gypaetus barbatus* | Bearded vulture | High alpine cliffs (reintroduced) |

To use a subset, uncomment `SPECIES` in the cell below and remove the lines you don't want.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# ── Load defaults from config/default.yaml ────────────────────────────────────
with open('config/default.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Override here — None = use value from default.yaml ────────────────────────

# Species to download: None = all 20 Alpine species from config/default.yaml
# To use a subset, uncomment SPECIES and remove the lines you don't want
SPECIES = None
# SPECIES = [
#     "Turdus torquatus",         # Ring ouzel              — rocky slopes, high-altitude forests
#     "Phoenicurus ochruros",     # Black redstart          — rocky terrain, mountain villages
#     "Prunella collaris",        # Alpine accentor         — high rocky areas above treeline
#     "Pyrrhocorax graculus",     # Yellow-billed chough    — alpine cliffs and glaciers
#     "Pyrrhocorax pyrrhocorax",  # Red-billed chough       — alpine meadows and cliffs
#     "Tichodroma muraria",       # Wallcreeper             — vertical rock faces
#     "Anthus spinoletta",        # Water pipit             — alpine meadows and streams
#     "Montifringilla nivalis",   # White-winged snowfinch  — above treeline, snowfields
#     "Lagopus muta",             # Rock ptarmigan          — high alpine tundra
#     "Dryocopus martius",        # Black woodpecker        — subalpine conifer forests
#     "Tetrao urogallus",         # Western capercaillie    — old-growth conifer forests
#     "Picoides tridactylus",     # Three-toed woodpecker   — spruce forests
#     "Loxia curvirostra",        # Common crossbill        — conifer forests
#     "Nucifraga caryocatactes",  # Spotted nutcracker      — mountain conifer forests
#     "Regulus ignicapilla",      # Firecrest               — mixed mountain forests
#     "Cinclus cinclus",          # White-throated dipper   — alpine streams and torrents
#     "Ficedula albicollis",      # Collared flycatcher     — deciduous mountain forests
#     "Saxicola rubetra",         # Whinchat                — subalpine meadows
#     "Emberiza cia",             # Rock bunting            — rocky slopes with sparse vegetation
#     "Gypaetus barbatus",        # Bearded vulture         — high alpine cliffs (reintroduced)
# ]

# Max recordings per species: None = 100 (from config)
MAX_PER_SPECIES = None
# MAX_PER_SPECIES = 50

# Quality filter: None = "A" (best only); set "all" for any grade (more data, lower quality)
QUALITY = None
# QUALITY = "all"

# Country filter: None = worldwide
COUNTRIES = None
# COUNTRIES = ["Italy", "Austria", "Switzerland"]

# Audio parameters: None = use value from config/default.yaml
SAMPLE_RATE    = None  # default: 22050 Hz
CLIP_DURATION  = None  # default: 5.0 s
CLIP_OVERLAP   = None  # default: 0.0 (no overlap)
N_MELS         = None  # default: 128
N_FFT          = None  # default: 2048
HOP_LENGTH     = None  # default: 512
F_MIN          = None  # default: 500.0 Hz
F_MAX          = None  # default: 15000.0 Hz
TOP_DB         = None  # default: 80.0 dB
IMG_SIZE       = None  # default: (224, 224) px

# ── Resolve: override wins over YAML default ──────────────────────────────────
d, a = cfg['download'], cfg['audio']

SPECIES         = SPECIES         or cfg['species']
MAX_PER_SPECIES = MAX_PER_SPECIES or d['max_per_species']
QUALITY         = QUALITY         or d['quality']
COUNTRIES       = COUNTRIES       or d.get('countries') or None
RAW_DIR         = cfg['data']['raw_dir']
PROCESSED_DIR   = cfg['data']['processed_dir']

from src.preprocessing import AudioConfig
AUDIO_CONFIG = AudioConfig(
    sample_rate   = SAMPLE_RATE   or a['sample_rate'],
    clip_duration = CLIP_DURATION or a['clip_duration'],
    clip_overlap  = CLIP_OVERLAP  if CLIP_OVERLAP is not None else a['clip_overlap'],
    n_mels        = N_MELS        or a['n_mels'],
    n_fft         = N_FFT         or a['n_fft'],
    hop_length    = HOP_LENGTH    or a['hop_length'],
    f_min         = F_MIN         or a['f_min'],
    f_max         = F_MAX         or a['f_max'],
    top_db        = TOP_DB        or a['top_db'],
    img_size      = IMG_SIZE      or tuple(a['img_size']),
)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"{'─'*50}")
print(f"  Species ({len(SPECIES)}):")
for s in SPECIES:
    print(f"    • {s}")
print(f"{'─'*50}")
print(f"  Max/species : {MAX_PER_SPECIES}  |  Quality: {QUALITY}  |  Countries: {COUNTRIES or 'worldwide'}")
print(f"  Sample rate : {AUDIO_CONFIG.sample_rate} Hz  |  Clip: {AUDIO_CONFIG.clip_duration}s  |  Overlap: {AUDIO_CONFIG.clip_overlap}")
print(f"  Mel bins    : {AUDIO_CONFIG.n_mels}  |  FFT: {AUDIO_CONFIG.n_fft}  |  Hop: {AUDIO_CONFIG.hop_length}")
print(f"  Freq range  : {AUDIO_CONFIG.f_min}–{AUDIO_CONFIG.f_max} Hz  |  Top dB: {AUDIO_CONFIG.top_db}")
print(f"  Image size  : {AUDIO_CONFIG.img_size[0]}×{AUDIO_CONFIG.img_size[1]} px")
print(f"{'─'*50}")

---
## 3. Download

Downloads `.mp3` recordings from the [Xeno-canto API v3](https://xeno-canto.org/explore/api).

> **API key required** since October 2025. Get a free key at [xeno-canto.org/article/854](https://xeno-canto.org/article/854) and set it as:
> ```bash
> export XENO_CANTO_API_KEY="your_key"
> ```
> If not set, you will be prompted interactively below.

Already-downloaded files are skipped automatically — safe to re-run.

In [ ]:
import pandas as pd
from src.download import XenoCantoDownloader

downloader = XenoCantoDownloader(output_dir=RAW_DIR)

results = downloader.download_species(
    species_list    = SPECIES,
    max_per_species = MAX_PER_SPECIES,
    quality         = QUALITY,
    countries       = COUNTRIES,
)

# Summary table
rows = [{'species': sp, 'downloaded': len(paths)} for sp, paths in results.items()]
df_dl = pd.DataFrame(rows)
print(f"\nTotal downloaded: {df_dl['downloaded'].sum()} files across {len(df_dl)} species")
df_dl

In [ ]:
# Verify files on disk and print directory tree
downloaded = downloader.list_downloaded()

summary_rows = []
for species_dir, files in downloaded.items():
    total_mb = sum(f.stat().st_size for f in files) / 1_048_576
    summary_rows.append({'species': species_dir, 'files': len(files), 'MB': round(total_mb, 2)})

df_disk = pd.DataFrame(summary_rows)
print(df_disk.to_string(index=False))
print(f"\nTotal: {df_disk['files'].sum()} files — {df_disk['MB'].sum():.1f} MB")

---
## 4. Preprocessing

Each `.mp3` recording is:
1. Loaded and resampled to `sample_rate` Hz
2. Sliced into `clip_duration`-second windows
3. Converted to a log-amplitude mel spectrogram
4. Saved as a `img_size` PNG — ready for CNN training

In [ ]:
# One spectrogram per species — visual sanity check before running the full pipeline
# Only shows species configured in section 2 (SPECIES), ignoring any leftover dirs
configured = {s.replace(' ', '_') for s in SPECIES}
species_dirs = sorted(
    d for d in Path(RAW_DIR).iterdir()
    if d.is_dir() and d.name in configured
)
representatives = [(d.name, sorted(d.glob('*.mp3'))[0]) for d in species_dirs if sorted(d.glob('*.mp3'))]
if not representatives:
    print('No .mp3 files found — check that step 3 completed successfully.')
else:
    cols = min(4, len(representatives))
    rows = (len(representatives) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes_flat = np.array(axes).flatten()
    for ax in axes_flat:
        ax.set_visible(False)
    for ax, (name, mp3) in zip(axes_flat, representatives):
        ax.set_visible(True)
        try:
            converter.plot_spectrogram(mp3, clip_index=0, ax=ax, title=name.replace('_', ' '))
        except Exception as e:
            ax.set_title(f'{name}\n(error)')
    fig.suptitle('Mel spectrogram — one per species', fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
# One spectrogram per species — visual sanity check before running the full pipeline
species_dirs = sorted([d for d in Path(RAW_DIR).iterdir() if d.is_dir()])
representatives = [(d.name, sorted(d.glob('*.mp3'))[0]) for d in species_dirs if sorted(d.glob('*.mp3'))]

if representatives:
    cols = min(4, len(representatives))
    rows = (len(representatives) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.5 * rows))
    axes_flat = np.array(axes).flatten()
    for ax in axes_flat:
        ax.set_visible(False)
    for ax, (name, mp3) in zip(axes_flat, representatives):
        ax.set_visible(True)
        try:
            converter.plot_spectrogram(mp3, clip_index=0, ax=ax, title=name.replace('_', ' '))
        except Exception as e:
            ax.set_title(f'{name}\n(error)')
    fig.suptitle('Mel spectrogram — one per species', fontsize=13, y=1.01)
    fig.tight_layout()
    plt.show()

In [ ]:
# Run full preprocessing pipeline
# Set OVERWRITE = True to regenerate existing images
OVERWRITE = False

summary = converter.process_all(input_dir=RAW_DIR, overwrite=OVERWRITE, species=SPECIES)

print('\nClips per species:')
for species, n_clips in sorted(summary.items()):
    print(f'  {species:<40} {n_clips:>4} clips')
print(f'\nTotal clips: {sum(summary.values())}')

---
## 5. Dataset summary

In [ ]:
processed_path = Path(PROCESSED_DIR)
species_counts = {
    d.name: len(list(d.glob('*.png')))
    for d in sorted(processed_path.iterdir())
    if d.is_dir() and list(d.glob('*.png'))
}
total = sum(species_counts.values())

print(f'Species : {len(species_counts)}')
print(f'Clips   : {total}')
print(f'Img size: {AUDIO_CONFIG.img_size[0]}×{AUDIO_CONFIG.img_size[1]} px')
print(f'Duration: {AUDIO_CONFIG.clip_duration}s per clip @ {AUDIO_CONFIG.sample_rate} Hz')

if species_counts:
    fig, ax = plt.subplots(figsize=(max(8, len(species_counts) * 0.7), 5))
    names  = [n.replace('_', '\n') for n in species_counts]
    counts = list(species_counts.values())
    bars = ax.bar(range(len(names)), counts, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7)
    ax.set_ylabel('Number of clips')
    ax.set_title(f'Clip distribution — {len(species_counts)} species, {total} total')
    ax.bar_label(bars, fontsize=7, padding=2)
    fig.tight_layout()
    plt.show()

In [ ]:
# Random sample of processed spectrogram images
import random
from PIL import Image

processed_path = Path(PROCESSED_DIR)
species_counts = {
    d.name: len(list(d.glob('*.png')))
    for d in sorted(processed_path.iterdir())
    if d.is_dir() and list(d.glob('*.png'))
}
total = sum(species_counts.values())

png_files = list(processed_path.rglob('*.png'))
if not png_files:
    print('No PNG files found — check that step 4 completed successfully.')
else:
    sample = random.sample(png_files, min(12, len(png_files)))
    cols = 4
    rows = (len(sample) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    for ax in np.array(axes).flatten():
        ax.set_visible(False)
    for ax, p in zip(np.array(axes).flatten(), sample):
        ax.set_visible(True)
        ax.imshow(Image.open(p))
        ax.set_title(p.parent.name.replace('_', ' '), fontsize=7)
        ax.axis('off')
    fig.suptitle('Random sample — processed spectrogram images', fontsize=11)
    fig.tight_layout()
    plt.show()
    print(f'\nDataset ready: {total} images across {len(species_counts)} species in {PROCESSED_DIR}/')
    print('Next step: training notebook (coming).')